<hr>

# 🕸️ WEB SCRAPING - seloger.com


<style>
h1 {
    text-align: left;
    color: blue;
    font-weight: bold;
}

</style>
<hr>

In [ ]:
import time
import csv
from dataclasses import dataclass, asdict
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup


@dataclass
class Listing:
    title: str
    price: str
    url: str


class Scraper:
    def __init__(self, base_url, delay=2.0):
        self.base_url = base_url.rstrip("/")
        self.delay = delay
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (compatible; Scraper/1.0)"
        })

    def fetch(self, url):
        time.sleep(self.delay)
        response = self.session.get(url, timeout=15)
        response.raise_for_status()
        return response.text

    def parse_listings(self, html):
        soup = BeautifulSoup(html, "html.parser")
        results = []

        # 👇 CHANGE THESE SELECTORS for your target site
        cards = soup.select(".listing-card")

        for card in cards:
            title_el = card.select_one(".listing-title")
            price_el = card.select_one(".listing-price")
            link_el = card.select_one("a[href]")

            if not title_el or not link_el:
                continue

            results.append(Listing(
                title=title_el.get_text(strip=True),
                price=price_el.get_text(strip=True) if price_el else "",
                url=urljoin(self.base_url, link_el["href"])
            ))

        return results

    def get_next_page(self, html):
        soup = BeautifulSoup(html, "html.parser")

        # 👇 CHANGE selector if needed
        next_btn = soup.select_one("a.next")

        if next_btn:
            return urljoin(self.base_url, next_btn["href"])
        return None

    def scrape(self, start_url, max_pages=5):
        url = start_url
        all_results = []

        for _ in range(max_pages):
            print(f"Scraping: {url}")
            html = self.fetch(url)

            listings = self.parse_listings(html)
            all_results.extend(listings)

            next_url = self.get_next_page(html)
            if not next_url:
                break

            url = next_url

        return all_results


def save_to_csv(data, filename="results.csv"):
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["title", "price", "url"])
        writer.writeheader()
        for item in data:
            writer.writerow(asdict(item))


if __name__ == "__main__":
    scraper = Scraper(base_url="https://example.com")

    results = scraper.scrape(
        start_url="https://example.com/search",
        max_pages=3
    )

    save_to_csv(results)
    print(f"Saved {len(results)} listings")